In [1]:
import cv2
import os
import subprocess

In [2]:
def cut_subvideo(video_path, start_time, end_time, output_path):
    """Uses FFmpeg to cut the subvideo from start_time to end_time."""
    command = [
        'ffmpeg', '-y', '-i', video_path, '-ss', str(start_time), '-to', str(end_time),
        '-c', 'copy', output_path
    ]
    subprocess.run(command)

In [ ]:
def SegmentVideo(video_path, output_dir, segment_duration=15):
    """
    Segments the video into chunks of the given duration (in seconds).
    
    Args:
    - video_path: Path to the video file.
    - output_dir: Directory to save the output video segments.
    - segment_duration: Duration of each segment in seconds (default is 15 seconds).
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    video_id = video_path.split('/')[-1].split('.')[0]
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total_duration = total_frames / fps  # Duration of the video in seconds
    
    segment_count = 0
    start_time = 0
    end_time = segment_duration

    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    while start_time < total_duration:
        # Set the output path for the current segment
        output_segment_path = os.path.join(output_dir, f'{video_id}_{start_time}_{min(end_time, total_duration)}.mp4')
        
        # Cut the subvideo from start_time to end_time
        cut_subvideo(video_path, start_time, min(end_time, total_duration), output_segment_path)
        segment_count += 1
        
        print(f'Segment saved from {start_time} to {min(end_time, total_duration)}')

        # Move to the next segment
        start_time = end_time
        end_time = start_time + segment_duration

    cap.release()
    print(f"Video segmented into {segment_count} segments.")

folder_path = '/kaggle/input/batch-3-videos/video'
output_dir = '/kaggle/working/'
for filename in os.listdir(folder_path):
    if filename.startswith('L26') and '301' <= filename[5:8] <= '400' and filename.endswith('.mp4'):
        video_path = os.path.join(folder_path, filename)
        SegmentVideo(video_path, output_dir)